---
## Phase 1 — Data Warehouse Design (Star Schema)

```
                    ┌─────────────────┐
                    │   Dim_Date      │
                    │  date_key (PK)  │
                    │  year · quarter │
                    │  month · week   │
                    └────────┬────────┘
                             │
  ┌─────────────┐   ┌────────▼────────┐   ┌─────────────┐
  │ Dim_Customer│   │  Fact_Orders    │   │ Dim_Product │
  │customer_key ├───┤  order_id (NK)  ├───┤ product_key │
  │segment      │   │  customer_key   │   │ category    │
  │country      │   │  product_key    │   │ brand       │
  │acq_channel  │   │  date_key       │   │ unit_cost   │
  │rfm_segment  │   │  country_key    │   │ list_price  │
  └─────────────┘   │  channel_key    │   └─────────────┘
                    │  ─────────────  │
                    │  net_revenue    │   ┌─────────────┐
                    │  gross_profit   │   │ Dim_Country │
                    │  discount_amt   ├───┤ country_key │
                    │  is_return      │   │ iso_code    │
                    └────────┬────────┘   │ region      │
                             │            └─────────────┘
                    ┌────────▼────────┐
                    │  Dim_Channel    │
                    │  channel_key    │
                    │  channel_name   │
                    │  payment_method │
                    └─────────────────┘
```

| Table | Grain | Key type |
|---|---|---|
| Fact_Orders | 1 row per order line | Surrogate + natural keys |
| Fact_Returns | 1 row per return event | FK to Fact_Orders |
| Dim_Customer | 1 row per customer | SCD Type 2 for RFM segment |
| Dim_Product | 1 row per product | Surrogate key |
| Dim_Date | 1 row per calendar day | Integer YYYYMMDD |
| Dim_Country | 1 row per country | Surrogate key |
| Dim_Channel | 1 row per channel | Surrogate key |


---
## Phase 2 — dbt Project Structure

```
lumiere/
├── dbt_project.yml          # project config + materialisation defaults
├── profiles.yml             # PostgreSQL connection (git-ignored)
├── packages.yml             # dbt-utils, dbt-expectations
└── models/
    ├── staging/             # raw → typed, renamed  (materialised: view)
    │   ├── sources.yml      # source declarations + column-level tests
    │   ├── schema.yml       # staging model tests
    │   ├── stg_orders.sql
    │   ├── stg_products.sql
    │   ├── stg_customers.sql
    │   ├── stg_returns.sql
    │   └── stg_sales_targets.sql
    ├── intermediate/        # business logic  (materialised: table)
    │   ├── int_orders_enriched.sql   # joins + margin + date dims
    │   └── int_customer_rfm.sql      # RFM scores via window functions
    └── marts/               # Tableau-ready  (materialised: table)
        ├── fct_orders.sql
        ├── fct_returns.sql
        ├── mart_commercial_kpis.sql
        ├── mart_customer_kpis.sql
        └── mart_country_targets.sql
```

### Key dbt tests applied

| Test | Column | Model |
|---|---|---|
| `unique` | `order_id` | stg_orders |
| `not_null` | `customer_id`, `product_id` | stg_orders |
| `accepted_values` | `channel` | stg_orders |
| `relationships` | `customer_id → stg_customers` | stg_orders |
| `expression_is_true` | `discount_pct BETWEEN 0 AND 0.5` | stg_orders |
| `unique` | `product_id` | stg_products |
| `unique` | `customer_id` | stg_customers |


### 1. Connect to PostgreSQL

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# ─── CONFIG ──────────────────────────────────────────────────────────────────
DB_URL = os.getenv("LUMIERE_DB_URL")
SOURCE_FILE = "/Users/ivanacaridad/Documents/GitHub/lumiere_analytics/data/raw/Lumiere_EU_Ecommerce.xlsx"
RAW_SCHEMA  = "raw"
TABLES = {
    "orders":        "Orders",
    "products":      "Products",
    "customers":     "Customers",
    "returns":       "Returns",
    "sales_targets": "Sales Targets",
}

In [3]:
# define a function to convert column names to snake_case
def snake_case(col: str) -> str:
    return col.strip().lower().replace(" ", "_")

### 2. Load to SQL 'orders_enriched', 'returns', etc

In [4]:
# define a function to load each sheet into a pandas DataFrame and then to Postgres
def load_to_postgres(engine, df: pd.DataFrame, table: str):
    """ Load a df to Postgres, replacing the table if it already exists. """
    df.columns = [snake_case(col) for col in df.columns]
    df.to_sql(
        name=table,
        con=engine, 
        schema=RAW_SCHEMA,
        if_exists="replace",
        index=False,
        chunksize=5000,                         #chunk size for batch insert
        method="multi",                         #multi-row insert for better performance
    )
    print(f" {table}: {len(df):,} rows loaded.")

In [5]:
def run_post_load_checks(engine):
    """ Quick check after loading."""
    checks = {
        "orders": "SELECT COUNT(*) FROM raw.orders;",
        "products": "SELECT COUNT(*) FROM raw.products;",
        "customers": "SELECT COUNT(*) FROM raw.customers;",
        "returns": "SELECT COUNT(*) FROM raw.returns;",
        "sales_targets": "SELECT COUNT(*) FROM raw.sales_targets;",
    }
    print("\n[Post-load row counts]")
    with engine.connect() as conn:
        for table, sql in checks.items():
            count = conn.execute(text(sql)).scalar() #scalar() to get single value result
            print(f" {table}: {count:,}")

In [6]:
def main():
    # Build SQLAlchemy engine from the DB_URL environment variable
    engine = create_engine(DB_URL, pool_pre_ping=True)

    # Ensure the raw schema exists before loading any tables
    with engine.connect() as conn:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA};"))
        conn.commit()
    print(f"[INFO] Schema '{RAW_SCHEMA}' is ready.")

    # Read all Excel sheets into a dict of DataFrames at once
    xl = pd.read_excel(SOURCE_FILE, sheet_name=None)
    print(f"\n[INFO] Loading {len(TABLES)} tables to PostgreSQL...")

    # Loop through each table mapping and upload to Postgres
    for pg_table, sheet_name in TABLES.items():
        df = xl[sheet_name].copy()
        load_to_postgres(engine, df, pg_table)

    # Verify row counts after loading
    run_post_load_checks(engine)
    print("\n✅  All tables loaded. Run 'dbt run' next.")


# Call main() at the module level so it actually executes when this cell runs
main()

[INFO] Schema 'raw' is ready.

[INFO] Loading 5 tables to PostgreSQL...
 orders: 82,000 rows loaded.
 products: 177 rows loaded.
 customers: 5,200 rows loaded.
 returns: 6,150 rows loaded.
 sales_targets: 192 rows loaded.

[Post-load row counts]
 orders: 82,000
 products: 177
 customers: 5,200
 returns: 6,150
 sales_targets: 192

✅  All tables loaded. Run 'dbt run' next.
